In [1]:
import json
import pandas as pd
from pathlib import Path
DATA_PATH = DATA_PATH = Path("/home/eric/TCGA_data")

Read in the metadata from GDC to algin labels with WSIs

In [22]:
with open(DATA_PATH / "metadata.repository.2026-06-04.json") as f:
    metadata = json.load(f)

slides = []
for entry in metadata:
    for entity in entry["associated_entities"]:
        slides.append({
            "patient_name": entity["entity_submitter_id"],
            "sample_id": entity["entity_submitter_id"][:15],
            "entity_id": entity["entity_id"],
            "file_id": entry["file_id"],
            "file_name": entry["file_name"]
        })

slides_df = pd.DataFrame(slides)
clinical = pd.read_csv(DATA_PATH / "TCGA.BRCA.sampleMap_BRCA_clinicalMatrix", sep="\t")

Filter by PAM50 RNAseq classifications

In [33]:
key = "PAM50Call_RNAseq"

pam50 = clinical[["sampleID", key]].dropna()
pam50 = pam50[pam50[key] != "Normal"]

result = slides_df.merge(pam50, left_on="sample_id", right_on="sampleID", how="inner")
result = result[["patient_name", "entity_id", "file_id", key]].rename(columns={key: "subtype"})

# print(result)
print(f"{key} subtypes")
print(result.shape)
print(result["subtype"].value_counts())

PAM50Call_RNAseq subtypes
(841, 4)
subtype
LumA     425
LumB     214
Basal    138
Her2      64
Name: count, dtype: int64


Filter by Nature 2012 classifications

In [34]:
key = "PAM50_mRNA_nature2012"

pam50_nature = clinical[["sampleID", key]].dropna()
pam50_nature = pam50_nature[pam50_nature[key] != "Normal-like"]

result_2012 = slides_df.merge(pam50_nature, left_on="sample_id", right_on="sampleID", how="inner")
result_2012 = result_2012[["patient_name", "entity_id", "file_id", key]].rename(columns={key: "subtype"})

print(f"{key} subtypes")
print(result_2012.shape)
print(result_2012["subtype"].value_counts())


PAM50_mRNA_nature2012 subtypes
(508, 4)
subtype
Luminal A        230
Luminal B        128
Basal-like        95
HER2-enriched     55
Name: count, dtype: int64


Select 200 slides with preliminary PAM50 classifications

In [ ]:
# Proportional sampling of 200 slides
total = 200
class_counts = result["subtype"].value_counts()
class_proportions = (class_counts / class_counts.sum() * total).round().astype(int)
diff = total - class_proportions.sum()
class_proportions.iloc[0] += diff

sampled = pd.concat([
    group.sample(n=class_proportions[name], random_state=42)
    for name, group in result.groupby("subtype")
]).reset_index(drop=True)

print(f"Total sampled: {len(sampled)}")
print(sampled["subtype"].value_counts())

full_manifest = pd.read_csv(DATA_PATH / "gdc_manifest.tsv.txt", sep="\t")
manifest_200 = full_manifest[full_manifest["id"].isin(sampled["file_id"])]
print(f"Matched in manifest: {len(manifest_200)}")
manifest_200.to_csv(DATA_PATH / "manifest_200.tsv", sep="\t", index=False)

Total sampled: 200
subtype
LumA     101
LumB      51
Basal     33
Her2      15
Name: count, dtype: int64
Matched in manifest: 200
